# Week 5 — Prototype cross-encoder reranking

Smallest possible end-to-end check: an 8-passage toy corpus, a dense first stage (SBERT bi-encoder, same as W4) returning a top-K, then a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) reranking that top-K. Numbers here are diagnostic only — they verify the pipeline runs, they are **not** benchmark numbers.

The honest baseline lives in `experiments/run_reranker.py` (reranks the W4 dense top-100 over all 6,980 dev/small queries); see `outputs/week05_reranker/metrics.json` and `reports/generated/week05_reranker.md` for the real numbers.

**Narrative continuity**:
- W2 (BM25): lexical first stage. Cheap, but loses on paraphrase.
- W4 (Dense): semantic first stage. Pushes Recall@100 near ceiling.
- **W5 (Cross-encoder rerank)**: once recall has saturated, the remaining gain is *local ordering* — pushing the relevant passage from rank 3 to rank 1. That's what the CE does.

## Setup

In [ ]:
import sys
from pathlib import Path


def _find_project_root(markers=(".git", "pyproject.toml", "requirements.txt")):
    p = Path.cwd().resolve()
    for parent in (p, *p.parents):
        if any((parent / m).exists() for m in markers):
            return parent
    return p


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# macOS libomp workaround: faiss-cpu and torch each ship libomp.dylib.
# Loading both in the same process aborts/segfaults unless we tell the
# runtime it's fine. Must be set before the first faiss/torch import.
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
from src.retrieval.dense import DenseRetriever
from src.reranking.cross_encoder import CrossEncoderReranker

## 1. Toy corpus

Eight short passages designed so the bi-encoder will rank some topically-related-but-wrong passages above the true answer, leaving room for the cross-encoder to fix the order.

In [ ]:
corpus = [
    "The Eiffel Tower was completed in 1889 for the Paris World's Fair.",        # d0 — tower-topical, doesn't answer 'where'
    "The Eiffel Tower is located on the Champ de Mars in Paris, France.",          # d1 — directly answers 'where'
    "Paris is the capital of France and a major European city.",                   # d2 — paris-topical, no tower
    "Gustave Eiffel was a French civil engineer best known for the Eiffel Tower.", # d3 — tower-topical, no location
    "The CN Tower in Toronto is a famous communications tower in Canada.",         # d4 — tower-related distractor
    "France is a country in western Europe known for its cuisine and culture.",    # d5 — france-topical, no tower
    "Many tourists visit Paris each year to see iconic landmarks.",                # d6 — paris-topical, no tower
    "Photosynthesis converts sunlight into chemical energy in green plants.",      # d7 — fully off-topic
]
doc_ids = [f"d{i}" for i in range(len(corpus))]

## 2. Dense first stage

Same encoder as W4 (`all-MiniLM-L6-v2`). For this query the relevant doc is **d1** — it's the only passage that actually states the tower's location.

In [ ]:
dense = DenseRetriever(model_name="sentence-transformers/all-MiniLM-L6-v2", device="cpu")
dense.build(corpus, doc_ids)

query = "where is the eiffel tower located"
scores, ids = dense.retrieve(query, k=5)
print(f"Dense top-5 for: {query!r}")
for rank, (s, d) in enumerate(zip(scores, ids), 1):
    mark = "  ✓" if d == "d1" else "   "
    print(f"  {rank}. {d}{mark}  score={s:.3f}  {corpus[int(d[1:])][:80]}")

The dense retriever gets d1 to rank 1, but with a fairly thin margin over d3 (Gustave Eiffel) and d0 (history of the tower) — passages that are topically about the Eiffel Tower but don't actually answer *"where"*. That thin margin is exactly what hurts MRR/nDCG at scale: on harder queries the bi-encoder will tip the wrong way.

## 3. Cross-encoder rerank

Wrap the dense top-K in the reranker. The cross-encoder reads the *pair* (query, passage) jointly, so it can detect fine-grained alignment ("located on the Champ de Mars in Paris" actually answers "where is") that a bi-encoder, which sees the two sides separately, tends to miss.

In [ ]:
reranker = CrossEncoderReranker(
    model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu",
    batch_size=16,
)

candidates = [(d, corpus[int(d[1:])]) for d in ids]
reranked, info = reranker.rerank_batch([query], [candidates], show_progress_bar=False)

print(f"Reranker scored {info['n_pairs']} pairs in {info['score_seconds']:.2f} s")
print(f"Reranked top-5 for: {query!r}")
for rank, (d, s) in enumerate(reranked[0], 1):
    mark = "  ✓" if d == "d1" else "   "
    print(f"  {rank}. {d}{mark}  ce_score={s:+.3f}  {corpus[int(d[1:])][:80]}")

Two things to notice:

1. d1 is still rank 1, but the **score margin** is now huge — d1 (~+10) vs the next candidate (~+4) is roughly a 100× larger gap than the dense margin (0.819 vs 0.736). When MRR / nDCG are averaged over thousands of queries, that confidence translates directly into fewer top-1 errors.
2. The CE re-ordered the *runners-up*: the historical-fact passage (d0) outranks the biographical passage (d3) because it at least mentions Paris. The bi-encoder had it the other way around. On harder queries this is exactly the kind of fix that turns a rank-3 hit into a rank-1 hit.

## 4. Before vs after — side-by-side table

Compact view of how the rerank moved each candidate.

In [ ]:
dense_rank = {d: i + 1 for i, d in enumerate(ids)}
rerank_rank = {d: i + 1 for i, (d, _) in enumerate(reranked[0])}
print(f"  {'doc':4s}  {'dense_rank':>10s}  {'rerank_rank':>11s}  {'Δ':>4s}  passage")
for d in ids:
    delta = dense_rank[d] - rerank_rank[d]
    arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "·")
    print(f"  {d:4s}  {dense_rank[d]:>10d}  {rerank_rank[d]:>11d}  {arrow}{abs(delta):>2d}  {corpus[int(d[1:])][:60]}")

## 5. Why a cross-encoder helps once recall is saturated

- Bi-encoder (W4): encodes query and passage **independently**, then compares with a dot product. Fast (one matmul over the FAISS index) but loses the chance to attend across the two sides. Great for recall, weak on fine-grained ordering.
- Cross-encoder (W5): concatenates query + passage and runs a full transformer over the pair, with cross-attention between every query and passage token. Slow (O(K) forward passes per query, no index possible) but captures exactly the kind of *direct-answer* alignment the bi-encoder can't.

Production retrievers stack the two: bi-encoder for cheap top-100 recall, cross-encoder for precise top-10 ordering. That's what W5 implements on the official run.

## Limitations

- 8-passage corpus, 1 hand-written query: a smoke test, not an evaluation.
- `ms-marco-MiniLM-L-6-v2` is the *published* baseline weights, used as-is. No fine-tuning, no calibration.
- Latency numbers here are meaningless at this scale — the CE encodes 5 pairs in well under a second.

## Next — official reranking baseline

Rerank the W4 dense top-100 over the full dev/small query set:

```bash
python experiments/run_reranker.py
python -m src.reporting.build_report --week week05
```
Full 6,980 queries × top-100 on CPU is ~6 hours; pass `--num-eval-queries 1000` for a ~50-minute subsample with stable MRR/nDCG estimates.
